In [1]:
import numpy as np
import torch
import torch.distributions as dist
from mfg import MFG, MFG_config
from solver import make_example, detect_major_G, detect_major_G_closedform

In [2]:
# ── Simulation grid ──────────────────────────────────────────────────────────
T   = 1       # time horizon (Section 6)
Ndt = 512    # time steps  → dt = 0.001

# ── Major bank parameters (Fig B.5, Fig B.6) ─────────────────────────────────
G        = 0.5   # relative market size of major bank  (F = 1-G = 0.5)
a        = 5     # minor bank mean-reversion rate (Section 6.2 baseline)
a_0      = a * G # = 2.5  market-clearing condition (eq 2.8): a_0 = a*G
sigma_0  = 1.0   # major bank reserve volatility
q_0      = 1.0   # major bank incentive to trade with central bank  (Fig B.5)
epslon_0 = 10.0  # major bank running penalty on reserve deviation  (Fig B.6)
c_0      = 0.0   # major bank terminal penalty                       (Fig B.6)

# ── Minor bank parameters (Fig B.5) ──────────────────────────────────────────
sigma    = 1.0   # minor bank reserve volatility
q        = 1.0   # minor bank incentive to trade with central bank
epslon   = 1.5   # minor bank running penalty  (must satisfy q^2 <= epslon)
c        = 0.0   # minor bank terminal penalty

# ── Monte Carlo ───────────────────────────────────────────────────────────────
N     = 512     # number of minor banks (Section 6)
N_sim = 300  # Monte Carlo paths     (Section 6)

cfg = MFG_config(
    T=T, Ndt=Ndt,
    a_0=a_0, sigma_0=sigma_0, c_0=c_0, epslon_0=epslon_0, q_0=q_0,
    a=a,     sigma=sigma,     c=c,     epslon=epslon,       q=q,
    G=G,
)
print(cfg)
mfg = MFG(cfg)
X, x_bar_obs, true_idx = make_example(mfg, N)


MFG_config: {'T': 1, 'Ndt': 512, 'dt': 0.001953125, 'a_0': 2.5, 'sigma_0': 1.0, 'c_0': 0.0, 'epslon_0': 10.0, 'q_0': 1.0, 'a': 5, 'sigma': 1.0, 'c': 0.0, 'epslon': 1.5, 'q': 1.0, 'G': 0.5, 'F': 0.5}


In [3]:
print("\n=== EM relaxation (Tier 3, mean field unobserved -- estimated from X) ===")
prob3, G_hat3, step = detect_major_G(mfg, X, true_major_idx=true_idx, n_em_iters=100,
    n_inner_E_steps=5,
    n_inner_M_steps=5, verbose=True, fix_phi0=False,lam_entropy=50.0)
pred3 = int(prob3.argmax().item())
print(f"predicted={pred3}  true={true_idx}  correct={pred3==true_idx}  "
        f"G_hat={G_hat3.item():.4f}  G_true={cfg.G}")
print("top-3 prob:", np.round(np.sort(prob3.numpy())[::-1][:3], 3))



=== EM relaxation (Tier 3, mean field unobserved -- estimated from X) ===
  EM iter   0  J=4.4661e+05  G=0.3797  argmax=100
  EM iter   1  J=4.4662e+05  G=0.4400  argmax=100
  EM iter   2  J=4.4663e+05  G=0.5022  argmax=163
  EM iter   3  J=4.4664e+05  G=0.5644  argmax=163
  EM iter   4  J=4.4665e+05  G=0.6246  argmax=163
  EM iter   5  J=4.4666e+05  G=0.6811  argmax=163
  EM iter   6  J=4.4666e+05  G=0.7328  argmax=443
  EM iter   7  J=4.4667e+05  G=0.7788  argmax=443
  EM iter   8  J=4.4667e+05  G=0.8189  argmax=443
  EM iter   9  J=4.4667e+05  G=0.8524  argmax=443
  EM iter  10  J=4.4668e+05  G=0.8791  argmax=45
  EM iter  11  J=4.4668e+05  G=0.8999  argmax=369
  EM iter  12  J=4.4668e+05  G=0.9160  argmax=369
  EM iter  13  J=4.4668e+05  G=0.9285  argmax=369
  EM iter  14  J=4.4668e+05  G=0.9385  argmax=369
  EM iter  15  J=4.4668e+05  G=0.9465  argmax=369
  EM iter  16  J=4.4668e+05  G=0.9529  argmax=369
  EM iter  17  J=4.4668e+05  G=0.9581  argmax=216
  EM iter  18  J=4.4668e+0

# comparison test

In [ ]:

G_true_values = [0.1, 0.3, 0.5, 0.7, 0.9]
n_seeds = 50

rows = []
for G_true in G_true_values:
    cfg = MFG_config(T=1, Ndt=512,
                      a_0=a * G_true, sigma_0=sigma_0, c_0=c_0, epslon_0=epslon_0, q_0=q_0,
                      a=a, sigma=sigma, c=c, epslon=epslon, q=q,
                      G=G_true)
    mfg = MFG(cfg)
    mfg.solve_ODE()

    G_hats, correct_flags = [], []
    for seed in range(n_seeds):
        X, x_bar_obs, true_idx = make_example(mfg, N, seed=seed)
        prob, G_hat = detect_major_G(
            mfg, X, true_major_idx=true_idx,
            n_em_iters=100, n_inner_E_steps=5, n_inner_M_steps=5,
            verbose=False, fix_phi0=False, lam_entropy=50.0,
        )
        G_hats.append(G_hat.item())
        correct_flags.append(int(prob.argmax()) == true_idx)
        print(f"seed={seed:3d}")
              

    G_hats = np.array(G_hats)
    rows.append({
        "G_true": G_true,
        "mean_G_hat": G_hats.mean(),
        "rmse": np.sqrt(np.mean((G_hats - G_true) ** 2)),
        "acc_w": np.mean(correct_flags),
    })

print(f"{'G_true':>8} | {'mean G*':>8} | {'RMSE':>8} | {'acc(w)':>8}")
print("-" * 42)
for r in rows:
    print(f"{r['G_true']:>8.2f} | {r['mean_G_hat']:>8.4f} | {r['rmse']:>8.4f} | {r['acc_w']:>8.2%}")

In [ ]:
print("=== Closed-form M-step (Tier 3, mean field unobserved) ===")
prob_cf, G_hat_cf, hist_cf = detect_major_G_closedform(mfg, X,
    n_em_iters=30, n_inner_E_steps=20, verbose=True,
    true_major_idx=true_idx,lam_entropy=50.0,recalc_phi0=True)
pred_cf = int(prob_cf.argmax().item())
print(f"predicted={pred_cf}  true={true_idx}  correct={pred_cf==true_idx}  "
      f"G_hat={G_hat_cf.item():.4f}  G_true={cfg.G}")
print("top-3 prob:", np.round(np.sort(prob_cf.numpy())[::-1][:3], 3))
